# Spectral CT Dataset Preprocessing

## Standardized Format

This step provides a generic framework to convert any spectral CT dataset (dual-energy, multi-energy, or hyperspectral) into a standardized format.



**Input:** Any spectral CT dataset with arbitrary channel names  
**Output:** Standardized directory structure:
```
dataset_name/
├── spectral_0/
├── spectral_1/
├── ...
└── spectral_N/
```

In [ ]:
import os
import os.path as osp
import sys
import shutil
from pathlib import Path
from typing import List, Dict, Optional, Union
from types import SimpleNamespace as NS
import argparse
import re
import glob
import json
import random
import copy

import numpy as np
import torch
import cv2
from tqdm import trange

import matplotlib.pyplot as plt
import tifffile
import tigre
import tigre.algorithms as algs
import scipy
import spekpy
# import tomopy.prep.stripe as stripe_removal  # TomoPy ring artifact removal

sys.path.append("../")
from r2_gaussian.utils.ct_utils import get_geometry_tigre
from r2_gaussian.initialize_pcd import init_pcd

### Configuration

In [548]:
# # --------- Pommegranate dataset --------- 
# # Input
# INPUT_DATASET_PATH = Path('../../data_raw/pomme')
# CHANNEL_MAPPING = ['noisy', 'good']
# FILE_INDEX_LENGTH = 6
# FILE_TYPE = '*.tif'


# # Output
# OUTPUT_BASE_DIR = Path('./output')
# OUTPUT_DATASET_NAME = 'pomme'


# # Spectral constants
# ENERGY_CHANNELS = [50.0, 70.0]
# FILTERS = [
#     {"material": None, "thickness": 0.0},
#     {"material": None, "thickness": 0.0}
# ]


# # Data processing
# LOG_TRANSFORM = True
# NORMALIZE_FACTOR = 3.0
# USE_DARK_FLAT_SCANS = True
# EXPOSURE_BALANCE = False
# EXPOSURE_TIMES = {'low': 125, 'mid': 125, 'high': 250}
# REF_CHAN = 'low'


# # Test / Train split parameters
# N_TRAIN = 75
# N_TEST = 100


# # Image resizing
# SUBSAMPLE = 3.0       # shrink
# UPSCALE_FACTOR = 1.0  # grow


# # Volume generation
# object_scale=50.0 / SUBSAMPLE
# proj_rescale=50.0
# nVoxel=[256,256,256]
# sVoxel=[2.0,2.0,2.0]
# offOrigin=[0.0,0.0,0.0]
# offDetector=[0.0,0.0]
# accuracy=0.5
# skip=1

# # PCD initialization
# args = {
#     "recon_method": "fdk",
#     "n_points": 100000,
#     "density_thresh": 0.05,
#     "density_rescale": 0.15,
#     "random_density_max": 1.0,
# }

In [549]:
# # --------- Quail dataset ---------
# # Input
# INPUT_DATASET_PATH = Path('../../data_raw/quail/2018-04-13_quail_chest_phantom_raw_data')
# CHANNEL_MAPPING = ['low', 'mid', 'high']
# FILE_INDEX_LENGTH = 4
# FILE_TYPE = '*.tif'


# # Output
# OUTPUT_BASE_DIR = Path('./output')
# OUTPUT_DATASET_NAME = 'quail'


# # Spectral constants
# ENERGY_CHANNELS = [50.0, 80.0, 120.0]
# FILTERS = [
#     {"material": None, "thickness": 0.0},
#     {"material": "Al", "thickness": 1.0},
#     {"material": "Cu", "thickness": 0.5}
# ]


# # Data processing
# LOG_TRANSFORM = True
# NORMALIZE_FACTOR = 3.0
# USE_DARK_FLAT_SCANS = False
# EXPOSURE_BALANCE = False
# EXPOSURE_TIMES = {'low': 125, 'mid': 125, 'high': 250}
# REF_CHAN = 'low'


# # Test / Train split parameters
# N_TRAIN = 75
# N_TEST = 100


# # Image resizing
# SUBSAMPLE = 1.0       # shrink
# UPSCALE_FACTOR = 1.0  # grow


# # Volume generation
# object_scale=10.0
# proj_rescale=50.0
# nVoxel=[256,256,256]
# sVoxel=[2.0,2.0,2.0]
# offOrigin=[0.0,0.0,0.0]
# offDetector=[0.0,0.0]
# accuracy=0.5
# skip=1


# # PCD initialization
# args = {
#     "recon_method": "fdk",
#     "n_points": 50000,
#     "density_thresh": 0.05,
#     "density_rescale": 0.15,
#     "random_density_max": 1.0,
# }

In [550]:
# --------- Lizard dataset ---------
# Input
INPUT_DATASET_PATH = Path('./output/lizard_temp')
CHANNEL_MAPPING = ['0', '1', '2']
FILE_INDEX_LENGTH = 4
FILE_TYPE = '*.tif'


# Output
OUTPUT_BASE_DIR = Path('./output')
OUTPUT_DATASET_NAME = 'lizard'


# Spectral constants
# ENERGY_CHANNELS = [20.3573, 32.8403, 45.323299999999996]
ENERGY_CHANNELS = [21.7443, 35.6143, 47.9586]
FILTERS = [
    {"type": "mono", "material": None, "thickness": 0.0},
    {"type": "mono", "material": None, "thickness": 0.0},
    {"type": "mono", "material": None, "thickness": 0.0}
]


# Data processing
LOG_TRANSFORM = False
NORMALIZE_FACTOR = 1.0
USE_DARK_FLAT_SCANS = False
EXPOSURE_BALANCE = False
EXPOSURE_TIMES = {'low': 125, 'mid': 125, 'high': 250}
REF_CHAN = 'low'


# Test / Train split parameters
N_TRAIN = 75
N_TEST = 100


# Image resizing
SUBSAMPLE = 1.0
UPSCALE_FACTOR = 4.0


# Volume generation
object_scale=150.0
proj_rescale=300.0
nVoxel=[256,256,256]
sVoxel=[2.0,2.0,2.0]
offOrigin=[0.0,0.0,0.0]
offDetector=[0.0,0.0]
accuracy=0.5
skip=1

# PCD initialization
args = {
    "recon_method": "fdk",
    "n_points": 50000,
    "density_thresh": 0.05,
    "density_rescale": 0.15,
    "random_density_max": 1.0,
}

In [551]:
# # --------- Mouse (hindlimb) dataset ---------
# # Input
# INPUT_DATASET_PATH = Path('./data/mouse_hindlimb')
# CHANNEL_MAPPING = ['0', '1', '2']
# FILE_INDEX_LENGTH = 4
# FILE_TYPE = '*.tif'


# # Output
# OUTPUT_BASE_DIR = Path('./output')
# OUTPUT_DATASET_NAME = 'mouse_hindlimb'


# # Data processing
# LOG_TRANSFORM = False
# NORMALIZE_FACTOR = 1.0
# USE_DARK_FLAT_SCANS = False
# EXPOSURE_BALANCE = False
# EXPOSURE_TIMES = {'low': 125, 'mid': 125, 'high': 250}
# REF_CHAN = 'low'


# # Test / Train split parameters
# N_TRAIN = 75
# N_TEST = 100


# # Image resizing
# SUBSAMPLE = 1.0
# UPSCALE_FACTOR = 4.0


# # Volume generation
# object_scale=60.0
# proj_rescale=200.0
# nVoxel=[256,256,256]
# sVoxel=[2.0,2.0,2.0]
# offOrigin=[0.0,0.0,0.0]
# offDetector=[0.0,0.0]
# accuracy=0.5
# skip=1

# # PCD initialization
# args = {
#     "recon_method": "fdk",
#     "n_points": 100000,
#     "density_thresh": 0.05,
#     "density_rescale": 0.15,
#     "random_density_max": 1.0,
# }

In [552]:
# # --------- Mouse (forelimb) dataset ---------
# # Input
# INPUT_DATASET_PATH = Path('./data/mouse_forelimb')
# CHANNEL_MAPPING = ['0', '1', '2']
# FILE_INDEX_LENGTH = 4
# FILE_TYPE = '*.tif'


# # Output
# OUTPUT_BASE_DIR = Path('./output')
# OUTPUT_DATASET_NAME = 'mouse_forelimb'


# # Data processing
# LOG_TRANSFORM = False
# NORMALIZE_FACTOR = 1.0
# USE_DARK_FLAT_SCANS = False
# EXPOSURE_BALANCE = False
# EXPOSURE_TIMES = {'low': 125, 'mid': 125, 'high': 250}
# REF_CHAN = 'low'


# # Test / Train split parameters
# N_TRAIN = 75
# N_TEST = 100


# # Image resizing
# SUBSAMPLE = 1.0
# UPSCALE_FACTOR = 4.0


# # Volume generation
# object_scale=75.0
# proj_rescale=75.0
# nVoxel=[256,256,256]
# sVoxel=[2.0,2.0,2.0]
# offOrigin=[0.0,0.0,0.0]
# offDetector=[0.0,0.0]
# accuracy=0.5
# skip=1

# # PCD initialization
# args = {
#     "recon_method": "fdk",
#     "n_points": 100000,
#     "density_thresh": 0.05,
#     "density_rescale": 0.15,
#     "random_density_max": 1.0,
# }

In [553]:
# # ---------  Iodine phantom dataset ---------
# # Input
# INPUT_DATASET_PATH = Path('./output/iodine_phantom_temp')
# CHANNEL_MAPPING = ['0', '1', '2']
# FILE_INDEX_LENGTH = 4
# FILE_TYPE = '*.tif'


# # Output
# OUTPUT_BASE_DIR = Path('./output')
# OUTPUT_DATASET_NAME = 'iodine_phantom'


# # Spectral constants
# ENERGY_CHANNELS = [21.9067197198409, 35.89185291276528, 49.87698610568966]
# FILTERS = [
#     {"material": None, "thickness": 0.0},
#     {"material": None, "thickness": 0.0},
#     {"material": None, "thickness": 0.0}
# ]


# # Data processing
# LOG_TRANSFORM = False
# NORMALIZE_FACTOR = 1.0
# USE_DARK_FLAT_SCANS = False
# EXPOSURE_BALANCE = False
# EXPOSURE_TIMES = {'low': 125, 'mid': 125, 'high': 250}
# REF_CHAN = 'low'


# # Test / Train split parameters
# N_TRAIN = 75
# N_TEST = 100


# # Image resizing
# SUBSAMPLE = 1.0
# UPSCALE_FACTOR = 4.0


# # Volume generation
# object_scale=10.0
# proj_rescale=30.0
# nVoxel=[256,256,256]
# sVoxel=[2.0,2.0,2.0]
# offOrigin=[0.0,0.0,0.0]
# offDetector=[0.0,0.0]
# accuracy=0.5
# skip=1

# # PCD initialization
# args = {
#     "recon_method": "fdk",
#     "n_points": 50000,
#     "density_thresh": 0.05,
#     "density_rescale": 0.15,
#     "random_density_max": 1.0,
# }

In [554]:
# Do not change
N_CHANNELS = len(CHANNEL_MAPPING)
CHANNELS = [f"spectral_{i}" for i in range(N_CHANNELS)]
INTERMEDIATE_DATASET_DIR = Path('./intermediate') / OUTPUT_DATASET_NAME
CONFIG_PATH = INTERMEDIATE_DATASET_DIR / 'spectral_0/config.txt'
OUTPUT_DATASET_DIR = OUTPUT_BASE_DIR / OUTPUT_DATASET_NAME

SPECTRA_DIR = OUTPUT_DATASET_DIR / 'spectra'
energy_out_dirs = {c: SPECTRA_DIR / c for c in CHANNELS}
for c in CHANNELS:
    energy_out_dirs[c].mkdir(parents=True, exist_ok=True)

(OUTPUT_DATASET_DIR / 'images').mkdir(parents=True, exist_ok=True)
(OUTPUT_DATASET_DIR / 'images_train').mkdir(parents=True, exist_ok=True)
(OUTPUT_DATASET_DIR / 'images_test').mkdir(parents=True, exist_ok=True)

### Conversion

In [569]:
def find_channel_directories() -> Dict[str, Path]:
    channel_dirs = {}
    all_dirs = [d for d in INPUT_DATASET_PATH.iterdir() if d.is_dir()]
    
    for channel_name in CHANNEL_MAPPING:
        matching_dirs = [d for d in all_dirs if channel_name.lower() in d.name.lower()]
        
        if matching_dirs:
            channel_dirs[channel_name] = matching_dirs[0]
            print(f"Found channel '{channel_name}': {matching_dirs[0].name}")
        else:
            raise ValueError(f"Could not find directory for channel '{channel_name}' in {INPUT_DATASET_PATH}")
    
    return channel_dirs


def create_normalized_copies(channel_dirs: Dict[str, Path]):
    # Create output directory
    INTERMEDIATE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    
    for idx, channel_name in enumerate(CHANNEL_MAPPING):
        source_dir = channel_dirs[channel_name]
        target_dir = INTERMEDIATE_DATASET_DIR / f'spectral_{idx}'
        
        # Remove existing directory if it exists
        if target_dir.exists():
            shutil.rmtree(target_dir)
        target_dir.mkdir(parents=True, exist_ok=True)
        
        # Copy and rename files
        source_files = sorted(list(source_dir.glob(FILE_TYPE)))
        
        for source_file in source_files:
            # Normalize filename
            name, ext = os.path.splitext(source_file.name)
            normalized_name = name[-FILE_INDEX_LENGTH:] + ext
            target_file = target_dir / normalized_name
            
            # Copy file
            shutil.copy2(source_file, target_file)
        
        # Copy config files (.txt) to spectral_X directory as 'config.txt'
        txt_files = list(source_dir.glob('*.txt'))
        if txt_files:
            for txt_file in txt_files:
                dest_path = target_dir / 'config.txt'
                shutil.copy2(txt_file, dest_path)
            print(f"Copied spectral_{idx} with {len(source_files)} normalized files + {len(txt_files)} config file(s)")
        else:
            print(f"Copied spectral_{idx} with {len(source_files)} normalized files")
    

print(f"Input:  {INPUT_DATASET_PATH}")
print(f"Output: {INTERMEDIATE_DATASET_DIR}")
print(f"Channels: {len(CHANNEL_MAPPING)} ({', '.join(CHANNEL_MAPPING)})")

# Step 1: Find channel directories
print("\n Step 1: Finding spectral channel directories")       
channel_dirs = find_channel_directories()
        
# Step 2: Create standardized structure
print(f"\n Step 2: Creating standardized structure")
create_normalized_copies(channel_dirs)
        
print(f"\n Conversion complete: {INTERMEDIATE_DATASET_DIR}")

Input:  output/lizard_temp
Output: intermediate/lizard
Channels: 3 (0, 1, 2)

 Step 1: Finding spectral channel directories
Found channel '0': 0
Found channel '1': 1
Found channel '2': 2

 Step 2: Creating standardized structure
Copied spectral_0 with 180 normalized files + 1 config file(s)
Copied spectral_1 with 180 normalized files + 1 config file(s)
Copied spectral_2 with 180 normalized files + 1 config file(s)

 Conversion complete: intermediate/lizard


## Dataset Normalization



In [556]:
def compute_file_lists() -> Dict[str, List[Path]]:
    file_lists = {}
    for idx, chan_name in enumerate(CHANNELS):
        folder = f"{INTERMEDIATE_DATASET_DIR}/spectral_{idx}"
        files = sorted(glob.glob(os.path.join(folder, '*.tif')))
        files = [f for f in files if not (os.path.basename(f).startswith('di') or os.path.basename(f).startswith('io'))]
        if not files:
            raise FileNotFoundError(f'No TIFF files found in {folder}')
        file_lists[chan_name] = files
    return file_lists

file_lists = compute_file_lists()

# Ensure all channels have the same number of projections
N_VIEWS = len(file_lists[CHANNELS[0]])
assert all(len(file_lists[c]) == N_VIEWS for c in CHANNELS), "Channel lengths mismatch."

print(f"Found {N_VIEWS} projections per channel")


# Infer native image shape (aspect ratio preserved)
sample_img = cv2.imread(file_lists[CHANNELS[0]][0], cv2.IMREAD_UNCHANGED)
H, W = int(sample_img.shape[0]), int(sample_img.shape[1])

if SUBSAMPLE != 1.0:
    H = int(H / SUBSAMPLE)
    W = int(W / SUBSAMPLE)

if UPSCALE_FACTOR != 1.0:
    H = int(H * UPSCALE_FACTOR)
    W = int(W * UPSCALE_FACTOR)

print("Image height + width:", H, W)

Found 180 projections per channel
Image height + width: 320 320


In [557]:
# Train/Test split (R2-Gaussian)
rng = np.random.default_rng(0)
if N_TRAIN >= N_VIEWS:
    train_indices = list(range(N_VIEWS))
else:
    train_indices = sorted(set(int(round(i * (N_VIEWS - 1) / float(N_TRAIN - 1)))
                               for i in range(N_TRAIN)))
remaining = [i for i in range(N_VIEWS) if i not in set(train_indices)]
if N_TEST > len(remaining): N_TEST = len(remaining)
test_indices = sorted(rng.choice(remaining, size=N_TEST, replace=False).tolist())

train_set = set(train_indices)
test_set  = set(test_indices)

print(len(train_set))
print(len(test_set))

75
100


In [558]:
# -------------------------
# I0 estimation (air) per channel
# -------------------------
def compute_I0_for_channel(file_list, top_pct=0.05):
    """Compute I0 (air intensity) from a list of files."""
    top_means = []
    for fp in file_list:
        img = cv2.imread(fp, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise IOError(f'Failed to read {fp}')
        img = img.astype(np.float32)
        flat = img.ravel()
        k = max(1, int(len(flat) * top_pct))
        thr = flat.max() if k == 1 else np.partition(flat, -k)[-k]
        top_means.append(flat[flat >= thr].mean())
    return float(np.mean(top_means))

def load_dark_flat_scans(channel_dir, dark_pattern='di*.tif', flat_pattern='io*.tif'):
    """Load dark and flat-field scans if available."""
    dark_files = sorted(list(channel_dir.glob(dark_pattern)))
    flat_files = sorted(list(channel_dir.glob(flat_pattern)))
    
    if not dark_files or not flat_files:
        return None, None
    
    # Load and average dark scans
    dark_imgs = [cv2.imread(str(f), cv2.IMREAD_UNCHANGED).astype(np.float32) for f in dark_files]
    dark = np.mean(dark_imgs, axis=0)
    
    # Load and average flat-field scans
    flat_imgs = [cv2.imread(str(f), cv2.IMREAD_UNCHANGED).astype(np.float32) for f in flat_files]
    flat = np.mean(flat_imgs, axis=0)
    
    return dark, flat

# Check if dark/flat scans are available and compute I0 or load calibration data
I0_values = []
dark_scans = {}
flat_scans = {}

if USE_DARK_FLAT_SCANS:
    print("Checking for dark/flat-field scans...")
    all_channels_have_calibration = True
    
    for idx, chan_name in enumerate(CHANNELS):
        folder = INTERMEDIATE_DATASET_DIR / f'spectral_{idx}'
        dark, flat = load_dark_flat_scans(folder)
        
        if dark is not None and flat is not None:
            dark_scans[chan_name] = dark
            flat_scans[chan_name] = flat
            # I0 is the mean flat-field value (air intensity)
            I0_values.append(float(np.mean(flat)))
            print(f"  {chan_name}: Found dark/flat scans")
        else:
            all_channels_have_calibration = False
            print(f"  {chan_name}: No dark/flat scans found")
            break
    
elif LOG_TRANSFORM:
    print("Using I0 estimation method...")
    for chan_name in CHANNELS:
        I0_values.append(compute_I0_for_channel(file_lists[chan_name], top_pct=0.05))
else:
    for chan_name in CHANNELS:
        I0_values.append(1.0)

print(f"I0 values: {I0_values}")
print(f"Normalization method: {'Dark/Flat calibration' if USE_DARK_FLAT_SCANS else 'I0 estimation'}")

I0 values: [1.0, 1.0, 1.0]
Normalization method: I0 estimation


In [559]:
# NORMALIZE_FACTOR = 0.0 

# for idx in range(N_VIEWS):
#     for i, chan_name in enumerate(CHANNELS):
#         fp = file_lists[chan_name][idx]
#         img = cv2.imread(fp, cv2.IMREAD_UNCHANGED).astype(np.float32)

#         if LOG_TRANSFORM:
#             if USE_DARK_FLAT_SCANS:
#                 # Apply dark/flat-field correction: (I - dark) / (flat - dark)
#                 dark = dark_scans[chan_name]
#                 flat = flat_scans[chan_name]
#                 img_corrected = (img - dark) / (flat - dark + 1e-8)
#                 img_corrected = np.clip(img_corrected, 1e-6, 1.0)
#                 log_proj = -np.log(img_corrected)
#             else:
#                 # Beer–Lambert log transform with per-channel I0
#                 I0 = float(I0_values[i])
#                 img_clipped = np.clip(img, 1e-6, I0)
#                 log_proj = -np.log(img_clipped / I0)

#             if EXPOSURE_BALANCE:
#                 channel = CHANNEL_MAPPING[i]
#                 log_proj *= (EXPOSURE_TIMES[REF_CHAN] / EXPOSURE_TIMES[channel])
#         else:
#             # Data is already log-transformed
#             log_proj = img.copy()
#             log_proj[log_proj < 0] = 0

#         p95 = float(np.nanpercentile(log_proj, 95))
#         if p95 > NORMALIZE_FACTOR:
#             NORMALIZE_FACTOR = p95

# if not np.isfinite(NORMALIZE_FACTOR) or NORMALIZE_FACTOR <= 0:
#     NORMALIZE_FACTOR = 1.0

# print(f'Global 95th-percentile max across all channels: {NORMALIZE_FACTOR:.6f}')

In [560]:
for i in range(N_VIEWS):
    idx_name = f"{i+1:04d}.npy"
    for ci, c in enumerate(CHANNELS):
        fp  = file_lists[c][i]
        img = cv2.imread(str(fp), cv2.IMREAD_UNCHANGED).astype(np.float32)

        if LOG_TRANSFORM:
            if USE_DARK_FLAT_SCANS:
                # Apply dark/flat-field correction: (I - dark) / (flat - dark)
                dark = dark_scans[c]
                flat = flat_scans[c]
                img_corrected = (img - dark) / (flat - dark + 1e-8)
                img_corrected = np.clip(img_corrected, 1e-6, 1.0)
                logp = -np.log(img_corrected)
            else:
                # Beer–Lambert log transform with per-channel I0
                I0c = I0_values[ci]
                img  = np.clip(img, 1e-6, I0c)
                logp = -np.log(img / I0c)
            
            if EXPOSURE_BALANCE:
                channel = CHANNEL_MAPPING[ci]
                logp *= (EXPOSURE_TIMES[REF_CHAN] / EXPOSURE_TIMES[channel])
        else:
            # Data is already log-transformed
            logp = img.copy()
            # Set negative values to 0
            logp[logp < 0] = 0

        out = np.clip(logp / (NORMALIZE_FACTOR + 1e-8), 0.0, 2.0).astype(np.float32)

        # Subsample (shrink image size)
        if SUBSAMPLE != 1.0:
            h_ori, w_ori = out.shape
            h_new, w_new = int(h_ori / SUBSAMPLE), int(w_ori / SUBSAMPLE)
            out = cv2.resize(out, [w_new, h_new])
        # Upscale (grow image size)
        if UPSCALE_FACTOR != 1.0:
            h_ori, w_ori = out.shape
            h_new, w_new = int(h_ori * UPSCALE_FACTOR), int(w_ori * UPSCALE_FACTOR)
            out = cv2.resize(out, [w_new, h_new], interpolation=cv2.INTER_LANCZOS4)

        full_path  = energy_out_dirs[c] / idx_name

        if ci == 0:
            full_path_root  = OUTPUT_DATASET_DIR / 'images' / idx_name
            train_path_root = OUTPUT_DATASET_DIR / 'images_train' / idx_name
            test_path_root  = OUTPUT_DATASET_DIR / 'images_test'  / idx_name

        np.save(full_path, out)
        if ci == 0:
            np.save(full_path_root, out)

        if i in train_set:
            if ci == 0:
                np.save(train_path_root, out)
        if i in test_set:
            if ci == 0:
                np.save(test_path_root, out)

print(f"[done] Wrote full + split .npy files into: {OUTPUT_DATASET_DIR}")


[done] Wrote full + split .npy files into: output/lizard


## Volume Generation

Compute pseudo-ground-truth volume from the lowest energy spectral channel (spectral_0)

In [570]:
# Read configuration
n_proj = None
angle_interval = None
angle_start = None
angle_last = None
DSD = None
DSO = None
dDetector = None
binning = None

# Track optional distance pieces so we can derive DSD if needed
source_to_sample_dist = None
sample_to_detector_dist = None
source_to_detector_dist = None

def _clean_config_value(raw_value):
    value = raw_value.strip()
    if len(value) >= 2 and value[0] == value[-1] and value[0] in {'"', "'"}:
        value = value[1:-1].strip()
    return value

with open(CONFIG_PATH, "r") as f:
    for config_line in f.readlines():
        
        line = config_line.strip()
        if ':' in line and '=' not in line.split(':')[0]:
            separator = ':'
        elif '=' in line:
            separator = '='
        else:
            continue
        
        # Split by separator
        parts = line.split(separator)
        if len(parts) < 2:
            continue
        
        key = parts[0].strip().lower()
        value = _clean_config_value(parts[-1])
        if not value:
            continue
        
        if "numberimages" in key or "angle_count" in key or "number projections" in key:
            n_proj = int(value)
        elif "angleinterval" in key or "angle_step" in key:
            angle_interval = float(value)
        elif "anglefirst" in key or "angle_start" in key or "start angle" in key:
            angle_start = float(value)
        elif "anglelast" in key or "angle_end" in key or "last angle" in key:
            angle_last = float(value)
        elif "distancesourcedetector" in key or "source_to_detector_dist" in key or key == "sdd":
            source_to_detector_dist = float(value)
        elif "distancesourceorigin" in key or "source_to_sample_dist" in key or key == "sod":
            source_to_sample_dist = float(value)
        elif "sample_to_detector_dist" in key:
            sample_to_detector_dist = float(value)
        elif (("pixelsize" in key and "pixelsizeunit" not in key) or "detector_pixels_size" in key or key == "pixel size") and "projection" not in key and "voxel" not in key:
            dDetector = float(value) * SUBSAMPLE / UPSCALE_FACTOR / 1000 * object_scale
        elif "binning" in key and "binned" not in key and "method" not in key:
            if 'x' in value:
                binning = float(value.split("x")[0])
            else:
                binning = float(value)

if source_to_sample_dist is not None:
    DSO = source_to_sample_dist / 1000 * object_scale
if source_to_detector_dist is not None:
    DSD = source_to_detector_dist / 1000 * object_scale
elif source_to_sample_dist is not None and sample_to_detector_dist is not None:
    DSD = (source_to_sample_dist + sample_to_detector_dist) / 1000 * object_scale

if angle_start is None:
    angle_start = 0.0
if angle_last is None:
    angle_last = angle_start + 360.0

# Calculate angle_interval if not provided but we have start, last, and n_proj
if angle_interval is None and angle_start is not None and angle_last is not None and n_proj is not None and n_proj > 1:
    angle_interval = (angle_last - angle_start) / (n_proj - 1)
    print(f"Calculated angle_interval: {angle_interval}")

# Update dDetector according to binning value (only for the old format where binning is separate from pixel size)
if binning and binning > 1:
    dDetector *= binning

# Endpoint-free angle model.  If the config stores the last acquired view as
# 358 deg for a 180-view / 2-deg full scan, the physical scan span is 360 deg.
config_span = float(angle_last - angle_start)
if angle_interval is not None and abs(config_span + float(angle_interval) - 360.0) < 1.0:
    total_angle = 360.0
elif abs(config_span - 360.0) < 1.0:
    total_angle = 360.0
else:
    total_angle = config_span
angles_deg = np.linspace(angle_start, angle_start + total_angle, N_VIEWS, endpoint=False, dtype=np.float32)
angles_rad = (angles_deg * np.pi / 180.0).astype(np.float32)
angles = angles_rad

projection_train_list = [
    {"file_path": os.path.join('images_train', f'{i+1:04d}.npy'), "angle": float(angles_rad[i])} for i in train_indices
]
projection_test_list = [
    {"file_path": os.path.join('images_test', f'{i+1:04d}.npy'), "angle": float(angles_rad[i])} for i in test_indices
]

nDetector = [int(H), int(W)]
sDetector = (np.array(nDetector) * dDetector).tolist()
bbox = np.array(
    [
        np.array(offOrigin) - np.array(sVoxel) / 2,
        np.array(offOrigin) + np.array(sVoxel) / 2,
    ]
).tolist()

scanner_cfg = {
    "mode": "cone",
    "DSD": DSD,
    "DSO": DSO,
    "nDetector": nDetector,
    "sDetector": sDetector,
    "nVoxel": nVoxel,
    "sVoxel": sVoxel,
    "dVoxel": (np.array(sVoxel) / np.array(nVoxel)).tolist(),
    "offOrigin": offOrigin,
    "offDetector": offDetector,
    "accuracy": accuracy,
    "totalAngle": float(total_angle),
    "startAngle": float(angle_start),
    "noise": True,
    "filter": None,
}

# Print parsed values for verification
print(f"n_proj: {n_proj}")
print(f"angle_start: {angle_start}")
print(f"angle_last: {angle_last}")
print(f"angle_interval: {angle_interval}")
print(f"endpoint-free total_angle: {total_angle}")
print(f"first/last generated angle: {angles_deg[0]:.6g}, {angles_deg[-1]:.6g}")
print(f"DSD: {DSD}")
print(f"DSO: {DSO}")
print(f"dDetector: {dDetector}")
print(f"binning: {binning}")
print(scanner_cfg)
print(scanner_cfg["nVoxel"])
print(scanner_cfg["sVoxel"])
print(scanner_cfg["dVoxel"])

n_proj: 180
angle_start: 0.0
angle_last: 360.0
angle_interval: 2.0
endpoint-free total_angle: 360.0
first/last generated angle: 0, 358
DSD: 90.3
DSO: 49.800000000000004
dDetector: 0.009375
binning: None
{'mode': 'cone', 'DSD': 90.3, 'DSO': 49.800000000000004, 'nDetector': [320, 320], 'sDetector': [3.0, 3.0], 'nVoxel': [256, 256, 256], 'sVoxel': [2.0, 2.0, 2.0], 'dVoxel': [0.0078125, 0.0078125, 0.0078125], 'offOrigin': [0.0, 0.0, 0.0], 'offDetector': [0.0, 0.0], 'accuracy': 0.5, 'totalAngle': 360.0, 'startAngle': 0.0, 'noise': True, 'filter': None}
[256, 256, 256]
[2.0, 2.0, 2.0]
[0.0078125, 0.0078125, 0.0078125]


In [571]:
# Minimal spectral reconstruction helpers used by this notebook.
# This replaces the standalone reconstruct_spectral_ct.py workflow.

def reconstruct_channel_volume(projs: np.ndarray, angles_rad: np.ndarray, geo, method: str = "fdk", cgls_iters: int = 60) -> np.ndarray:
    if method == "fdk":
        vol = algs.fdk(projs[:, ::-1, :], copy.deepcopy(geo), angles_rad)
    elif method == "cgls":
        vol, _ = algs.cgls(projs[:, ::-1, :], copy.deepcopy(geo), angles_rad, cgls_iters, computel2=True)
    else:
        raise ValueError(f"Unsupported reconstruction method: {method}")
    return np.transpose(np.asarray(vol, dtype=np.float32), (2, 1, 0))


def correct_volume_background(vol: np.ndarray, air_quantile: float = 0.20):
    """Subtract low-positive-tail air offset while preserving signed reconstruction values."""
    vol = np.asarray(vol, dtype=np.float32)
    finite = vol[np.isfinite(vol)]
    positive = finite[finite > 0.0]
    if positive.size:
        threshold = float(np.quantile(positive, air_quantile))
        air = positive[positive <= threshold]
    else:
        threshold = float(np.quantile(finite, air_quantile)) if finite.size else 0.0
        air = finite[finite <= threshold]
    offset = float(np.median(air)) if air.size else 0.0
    return (vol - offset).astype(np.float32), offset


def make_support_mask(spectral_vol: np.ndarray, support_quantile: float = 0.80, dilation_iters: int = 2, largest_components: int = 1) -> np.ndarray:
    ref = np.mean(np.asarray(spectral_vol, dtype=np.float32), axis=0)
    positive = ref[ref > 0.0]
    if positive.size == 0:
        return np.ones(ref.shape, dtype=bool)
    threshold = float(np.quantile(positive, support_quantile))
    support = ref > threshold

    try:
        from scipy import ndimage
        if largest_components and largest_components > 0:
            labels, n_labels = ndimage.label(support)
            if n_labels > largest_components:
                counts = np.bincount(labels.ravel())
                counts[0] = 0
                keep = np.argsort(counts)[-largest_components:]
                support = np.isin(labels, keep)
        if dilation_iters and dilation_iters > 0:
            support = ndimage.binary_dilation(support, iterations=dilation_iters)
    except Exception as exc:
        print(f"[support] scipy.ndimage unavailable; using undilated support mask ({exc!r}).")
    return support.astype(bool)


def project_channel_volume(vol: np.ndarray, angles_rad: np.ndarray, geo, projection_type: str = "Siddon") -> np.ndarray:
    vol_tigre = np.transpose(np.asarray(vol, dtype=np.float32), (2, 1, 0))
    projs_tigre = tigre.Ax(vol_tigre, copy.deepcopy(geo), angles_rad, projection_type=projection_type)
    return np.asarray(projs_tigre[:, ::-1, :], dtype=np.float32)


def sample_init_pcd_from_volume(vol: np.ndarray, scanner_cfg: dict, n_points: int = 50000, density_quantile: float = 0.90, density_rescale: float = 0.15, seed: int = 0, support_mask: np.ndarray | None = None) -> np.ndarray:
    rng = np.random.default_rng(seed)
    sample_vol = np.clip(np.asarray(vol, dtype=np.float32), 0.0, None)
    positive = sample_vol[sample_vol > 0.0]
    density_thresh = float(np.quantile(positive, density_quantile)) if positive.size else 0.0

    valid_mask = sample_vol > density_thresh
    if support_mask is not None:
        valid_mask &= support_mask.astype(bool)
    valid_indices = np.argwhere(valid_mask)
    if valid_indices.size == 0:
        valid_indices = np.argwhere(sample_vol > 0.0)
    if valid_indices.size == 0:
        valid_indices = np.indices(sample_vol.shape).reshape(3, -1).T

    selected = valid_indices[rng.choice(len(valid_indices), size=n_points, replace=len(valid_indices) < n_points)]
    off_origin = np.asarray(scanner_cfg["offOrigin"], dtype=np.float32)
    d_voxel = np.asarray(scanner_cfg["dVoxel"], dtype=np.float32)
    s_voxel = np.asarray(scanner_cfg["sVoxel"], dtype=np.float32)
    positions = selected.astype(np.float32) * d_voxel - s_voxel / 2.0 + off_origin
    densities = sample_vol[selected[:, 0], selected[:, 1], selected[:, 2]] * float(density_rescale)
    return np.concatenate([positions, densities[:, None]], axis=1).astype(np.float64)


def save_ply(path: Path, pcd: np.ndarray) -> None:
    with path.open("w") as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {len(pcd)}\n")
        f.write("property float x\nproperty float y\nproperty float z\nproperty float density\n")
        f.write("end_header\n")
        for point in pcd:
            f.write(f"{point[0]} {point[1]} {point[2]} {point[3]}\n")

In [ ]:
def is_finite(tif_path):
    """Check if all values in a TIFF file are finite."""
    img = tifffile.imread(tif_path)
    return np.isfinite(img).all()

EPS = 1e-6
random.seed(0)

# Use only non-infinite projections / angles from spectral_0 for the legacy scalar reference volume.
proj_dir = Path(INTERMEDIATE_DATASET_DIR) / 'spectral_0'
raw_files = sorted(glob.glob(str(proj_dir / '*.tif')))
proj_files = [f for f in raw_files if not (os.path.basename(f).startswith('di') or os.path.basename(f).startswith('io'))]
valid_pairs = [(idx, p) for idx, p in enumerate(proj_files) if is_finite(p)]
if len(valid_pairs) == 0:
    raise ValueError("All projections are non-finite. Check input data.")
valid_indices, proj_files = zip(*valid_pairs)

proj_files = list(proj_files)
angles_valid = angles[list(valid_indices)]

if USE_DARK_FLAT_SCANS:
    dark_files = sorted(glob.glob(os.path.join(INTERMEDIATE_DATASET_DIR, 'spectral_0', 'di*.tif')))
    flat_files = sorted(glob.glob(os.path.join(INTERMEDIATE_DATASET_DIR, 'spectral_0', 'io*.tif')))
    I_dark = np.mean([tifffile.imread(p).astype(np.float32) for p in dark_files], axis=0)
    I_flat = np.mean([tifffile.imread(p).astype(np.float32) for p in flat_files], axis=0)
else:
    I0 = I0_values[0]

projs = []
for proj_path in proj_files:
    I_raw = tifffile.imread(proj_path).astype(np.float32)

    if LOG_TRANSFORM:
        if USE_DARK_FLAT_SCANS:
            I_corr = I_raw - I_dark
            I0_corr = I_flat - I_dark
            log_proj = -np.log(I_corr / I0_corr)
        else:
            log_proj = -np.log(I_raw / I0)
    else:
        # Data is already log-transformed
        log_proj = I_raw.copy()
        
    # Scale by object_scale / proj_rescale to control intensity
    log_proj = log_proj / proj_rescale * object_scale
    log_proj[log_proj < 0] = 0
    
    # Subsample (resize)
    if SUBSAMPLE != 1.0:
        h_ori, w_ori = log_proj.shape
        h_new, w_new = int(h_ori / SUBSAMPLE), int(w_ori / SUBSAMPLE)
        log_proj = cv2.resize(log_proj, [w_new, h_new], interpolation=cv2.INTER_AREA)
    
    # Upscale (high-quality interpolation)
    if UPSCALE_FACTOR != 1.0:
        h_ori, w_ori = log_proj.shape
        h_new, w_new = int(h_ori * UPSCALE_FACTOR), int(w_ori * UPSCALE_FACTOR)
        log_proj = cv2.resize(log_proj, [w_new, h_new], interpolation=cv2.INTER_LANCZOS4)
    
    projs.append(log_proj)

projs = np.stack(projs, axis=0).astype(np.float32)

ct_gt_save_path = osp.join(OUTPUT_DATASET_DIR, "vol_gt.npy")
geo = get_geometry_tigre(scanner_cfg)
ct_gt = reconstruct_channel_volume(projs, angles_valid[::skip], geo, method="fdk")
ct_gt, ct_gt_background_offset = correct_volume_background(ct_gt, air_quantile=0.20)
print(f"Volume shape: {ct_gt.shape}")
print(f"Volume range: [{ct_gt.min():.4f}, {ct_gt.max():.4f}]")
print(f"Background offset: {ct_gt_background_offset:.6g}")

np.save(ct_gt_save_path, ct_gt) 
print(f"Data saved in {OUTPUT_DATASET_DIR}")

TIGRE parameters
-----
Geometry parameters
Distance from source to detector (DSD) = 90.3 mm
Distance from source to origin (DSO)= 49.800000000000004 mm
-----
Detector parameters
Number of pixels (nDetector) = [320 320]
Size of each pixel (dDetector) = [0.009375 0.009375] mm
Total size of the detector (sDetector) = [3. 3.] mm
-----
Image parameters
Number of voxels (nVoxel) = [256 256 256]
Total size of the image (sVoxel) = [2. 2. 2.] mm
Size of each voxel (dVoxel) = [0.0078125 0.0078125 0.0078125] mm
-----
Offset correction parameters
Offset of image from origin (offOrigin) = [0. 0. 0.] mm
Offset of detector (offDetector) = [0. 0. 0.] mm
-----
Auxillary parameters
Samples per pixel of forward projection (accuracy) = 0.5
(180, 320, 320)
Volume shape: (256, 256, 256)
Volume range: [0.0000, 1.0282]
Data saved in output/lizard


In [563]:
spectral_cfg = {
    "n_channels": N_CHANNELS,
    "I0": I0_values,
    "E": ENERGY_CHANNELS,
    "S": FILTERS
}

In [564]:
meta_data = {
    "spectral": spectral_cfg,
    "scanner": scanner_cfg,
    "ct": "vol_gt.npy",
    "radius": 1.0,
    "bbox": bbox,
    "proj_train": projection_train_list,
    "proj_test": projection_test_list,
    "cameras": projection_train_list,
}

os.makedirs(OUTPUT_DATASET_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DATASET_DIR, 'meta_data.json'), 'w') as f:
    json.dump(meta_data, f, indent=2)

print(f'Metadata saved to {os.path.join(OUTPUT_DATASET_DIR, "meta_data.json")}')
print('Done.')

Metadata saved to output/lizard/meta_data.json
Done.


### Optional spectral reconstruction and reprojection export

This block reconstructs one signed volume per spectral channel, applies a background correction and support mask, then forward-projects the masked volumes into a trainable projection dataset. This is the notebook-local replacement for the standalone reconstruction script.

In [ ]:
RUN_SPECTRAL_RECON_EXPORT = True
RECON_METHOD = "fdk"
RECON_CGLS_ITERS = 60
APPLY_RECON_BACKGROUND_CORRECTION = True
RECON_AIR_QUANTILE = 0.20
APPLY_RECON_SUPPORT_MASK = True
RECON_SUPPORT_QUANTILE = 0.80
RECON_SUPPORT_DILATION_ITERS = 2
RECON_SUPPORT_LARGEST_COMPONENTS = 1
RECON_DENSITY_QUANTILE = 0.90
RECON_DENSITY_RESCALE = 0.15
REPROJECTED_DATASET_NAME = f"{OUTPUT_DATASET_NAME}_reprojected_fdk_fixed"
RECON_OUTPUT_DIR = OUTPUT_DATASET_DIR / "recon_spectral_fdk_fixed"
REPROJECTED_OUTPUT_DIR = OUTPUT_BASE_DIR / REPROJECTED_DATASET_NAME


def _load_saved_channel_stack(channel_idx: int):
    channel_dir = SPECTRA_DIR / f"spectral_{channel_idx}"
    files = sorted(channel_dir.glob("*.npy"))
    if not files:
        raise FileNotFoundError(f"No saved projections found in {channel_dir}")
    return np.stack([np.load(path).astype(np.float32) for path in files], axis=0), [path.name for path in files]


def _filter_projection_entries(entries, generated_names, subdir):
    filtered = []
    for entry in entries:
        name = Path(entry["file_path"]).name
        if name in generated_names:
            new_entry = dict(entry)
            new_entry["file_path"] = os.path.join(subdir, name)
            filtered.append(new_entry)
    return filtered


if RUN_SPECTRAL_RECON_EXPORT:
    if REPROJECTED_OUTPUT_DIR.exists():
        shutil.rmtree(REPROJECTED_OUTPUT_DIR)
    RECON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (REPROJECTED_OUTPUT_DIR / "spectra").mkdir(parents=True, exist_ok=True)
    (REPROJECTED_OUTPUT_DIR / "images").mkdir(parents=True, exist_ok=True)
    (REPROJECTED_OUTPUT_DIR / "images_train").mkdir(parents=True, exist_ok=True)
    (REPROJECTED_OUTPUT_DIR / "images_test").mkdir(parents=True, exist_ok=True)
    for ci in range(N_CHANNELS):
        (REPROJECTED_OUTPUT_DIR / "spectra" / f"spectral_{ci}").mkdir(parents=True, exist_ok=True)

    geo = get_geometry_tigre(scanner_cfg)
    volumes = []
    background_offsets = []
    reference_names = None

    for ci in range(N_CHANNELS):
        channel_projs, channel_names = _load_saved_channel_stack(ci)
        if reference_names is None:
            reference_names = channel_names
        elif channel_names != reference_names:
            raise ValueError(f"Projection file list for spectral_{ci} does not match spectral_0")

        print(f"Reconstructing spectral_{ci}: projections={channel_projs.shape}, method={RECON_METHOD}")
        vol = reconstruct_channel_volume(channel_projs, angles[:channel_projs.shape[0]], geo, method=RECON_METHOD, cgls_iters=RECON_CGLS_ITERS)
        np.save(RECON_OUTPUT_DIR / f"vol_channel_{ci}_raw.npy", vol)

        if APPLY_RECON_BACKGROUND_CORRECTION:
            vol, offset = correct_volume_background(vol, air_quantile=RECON_AIR_QUANTILE)
        else:
            offset = 0.0
        background_offsets.append(offset)
        print(f"  background offset spectral_{ci}: {offset:.6g}")
        np.save(RECON_OUTPUT_DIR / f"vol_channel_{ci}.npy", vol)
        volumes.append(vol)

    spectral_vol = np.stack(volumes, axis=0).astype(np.float32)
    np.save(RECON_OUTPUT_DIR / "vol_spectral_background_corrected.npy", spectral_vol)

    support_mask = None
    if APPLY_RECON_SUPPORT_MASK:
        support_mask = make_support_mask(
            spectral_vol,
            support_quantile=RECON_SUPPORT_QUANTILE,
            dilation_iters=RECON_SUPPORT_DILATION_ITERS,
            largest_components=RECON_SUPPORT_LARGEST_COMPONENTS,
        )
        np.save(RECON_OUTPUT_DIR / "support_mask.npy", support_mask.astype(np.uint8))
        spectral_vol = (spectral_vol * support_mask[None, ...]).astype(np.float32)
        print(f"Applied support mask: {int(support_mask.sum())}/{support_mask.size} voxels kept")

    np.save(RECON_OUTPUT_DIR / "vol_spectral.npy", spectral_vol)
    vol_combined = spectral_vol.mean(axis=0).astype(np.float32)
    np.save(RECON_OUTPUT_DIR / "vol_combined.npy", vol_combined)
    np.save(REPROJECTED_OUTPUT_DIR / "vol_gt.npy", vol_combined)

    pcd = sample_init_pcd_from_volume(
        vol_combined,
        scanner_cfg,
        n_points=args["n_points"],
        density_quantile=RECON_DENSITY_QUANTILE,
        density_rescale=RECON_DENSITY_RESCALE,
        seed=0,
        support_mask=support_mask,
    )
    np.save(RECON_OUTPUT_DIR / "init_pcd.npy", pcd)
    np.save(REPROJECTED_OUTPUT_DIR / "init_pcd.npy", pcd)
    save_ply(RECON_OUTPUT_DIR / "init_pcd.ply", pcd)
    save_ply(REPROJECTED_OUTPUT_DIR / "init_pcd.ply", pcd)

    projected_channels = []
    for ci, vol in enumerate(spectral_vol):
        print(f"Forward-projecting spectral_{ci}: views={len(reference_names)}")
        projected_channels.append(project_channel_volume(vol, angles[:len(reference_names)], geo, projection_type="Siddon"))
    projected_spectral = np.stack(projected_channels, axis=0).astype(np.float32)
    np.save(RECON_OUTPUT_DIR / "projected_spectral.npy", projected_spectral)

    train_names = {Path(entry["file_path"]).name for entry in projection_train_list}
    test_names = {Path(entry["file_path"]).name for entry in projection_test_list}
    for view_idx, name in enumerate(reference_names):
        for ci in range(N_CHANNELS):
            arr = np.clip(projected_spectral[ci, view_idx], 0.0, None).astype(np.float32)
            np.save(REPROJECTED_OUTPUT_DIR / "spectra" / f"spectral_{ci}" / name, arr)
            if ci == 0:
                np.save(REPROJECTED_OUTPUT_DIR / "images" / name, arr)
                if name in train_names:
                    np.save(REPROJECTED_OUTPUT_DIR / "images_train" / name, arr)
                if name in test_names:
                    np.save(REPROJECTED_OUTPUT_DIR / "images_test" / name, arr)

    generated_names = set(reference_names)
    reprojected_meta = copy.deepcopy(meta_data)
    reprojected_meta["ct"] = "vol_gt.npy"
    reprojected_meta["proj_train"] = _filter_projection_entries(projection_train_list, generated_names, "images_train")
    reprojected_meta["proj_test"] = _filter_projection_entries(projection_test_list, generated_names, "images_test")
    reprojected_meta["cameras"] = reprojected_meta["proj_train"]
    reprojected_meta["projection_source"] = {
        "type": "notebook_reprojected_reconstruction",
        "source_dataset": str(OUTPUT_DATASET_DIR),
        "channels": list(range(N_CHANNELS)),
        "view_count": len(reference_names),
        "background_offsets": background_offsets,
        "support_quantile": RECON_SUPPORT_QUANTILE if APPLY_RECON_SUPPORT_MASK else None,
    }
    with open(REPROJECTED_OUTPUT_DIR / "meta_data.json", "w") as f:
        json.dump(reprojected_meta, f, indent=2)

    reconstruction_meta = {
        "source_dataset": str(OUTPUT_DATASET_DIR),
        "reprojected_dataset": str(REPROJECTED_OUTPUT_DIR),
        "method": RECON_METHOD,
        "background_offsets": background_offsets,
        "support_mask": None if support_mask is None else {
            "file": "support_mask.npy",
            "support_quantile": RECON_SUPPORT_QUANTILE,
            "dilation_iters": RECON_SUPPORT_DILATION_ITERS,
            "largest_components": RECON_SUPPORT_LARGEST_COMPONENTS,
        },
        "angles": {
            "start_deg": float(scanner_cfg["startAngle"]),
            "total_deg": float(scanner_cfg["totalAngle"]),
            "n_views": len(reference_names),
            "endpoint": False,
        },
    }
    with open(RECON_OUTPUT_DIR / "reconstruction_meta.json", "w") as f:
        json.dump(reconstruction_meta, f, indent=2)

    print(f"Saved spectral reconstruction stack to {RECON_OUTPUT_DIR / 'vol_spectral.npy'}")
    print(f"Saved trainable reprojected dataset to {REPROJECTED_OUTPUT_DIR}")
    print(f"  train images: {len(reprojected_meta['proj_train'])}")
    print(f"  test images : {len(reprojected_meta['proj_test'])}")

### PCD

In [565]:
# Load training projections and angles
proj_paths = [os.path.join(OUTPUT_DATASET_DIR, p['file_path']) for p in projection_train_list]
projs_train = np.stack([np.load(p) for p in proj_paths], axis=0)
angles_train = np.array([p['angle'] for p in projection_train_list])

print(f"Training projections shape: {projs_train.shape}")
print(f"Training angles shape: {angles_train.shape}")
print(f"Angles range: [{angles_train.min():.4f}, {angles_train.max():.4f}] radians")

save_path = osp.join(OUTPUT_DATASET_DIR, "init_pcd.npy")

Training projections shape: (75, 320, 320)
Training angles shape: (75,)
Angles range: [0.0000, 6.2483] radians


In [566]:
init_pcd(
    projs=projs_train,
    angles=angles_train,
    geo=geo,
    scanner_cfg=scanner_cfg,
    args=args,
    save_path=save_path,
)

Initialize point clouds with the volume reconstructed from fdk.
TIGRE parameters
-----
Geometry parameters
Distance from source to detector (DSD) = 90.3 mm
Distance from source to origin (DSO)= 49.800000000000004 mm
-----
Detector parameters
Number of pixels (nDetector) = [320 320]
Size of each pixel (dDetector) = [0.009375 0.009375] mm
Total size of the detector (sDetector) = [3. 3.] mm
-----
Image parameters
Number of voxels (nVoxel) = [256 256 256]
Total size of the image (sVoxel) = [2. 2. 2.] mm
Size of each voxel (dVoxel) = [0.0078125 0.0078125 0.0078125] mm
-----
Offset correction parameters
Offset of image from origin (offOrigin) = [0. 0. 0.] mm
Offset of detector (offDetector) = [0. 0. 0.] mm
-----
Auxillary parameters
Samples per pixel of forward projection (accuracy) = 0.5
(75, 320, 320)
Initialization saved in output/lizard/init_pcd.npy.


In [567]:
intermediate = Path('./intermediate')
if intermediate.exists():
    shutil.rmtree(intermediate)
    print(f"Removed intermediate dataset")

Removed intermediate dataset
